# Pneumonia Detection - Improved MobileNetV2

## Improvements over baseline (87.98% accuracy)
1. **Focal Loss** - Focuses on hard-to-classify cases, handles class imbalance better
2. **CBAM Attention** - Channel + Spatial attention to focus on pneumonia regions
3. **Stronger Augmentation** - More aggressive transforms for better generalization
4. **Better LR Scheduling** - ReduceLROnPlateau with patience

Note: CLAHE was tested but hurt performance on this dataset (over-normalization)

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from glob import glob

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_recall_fscore_support
)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# Auto-detect dataset path
kaggle_input = '/kaggle/input'
possible_paths = glob(os.path.join(kaggle_input, '**/chest_xray'), recursive=True)
if possible_paths:
    DATA_DIR = possible_paths[0]
else:
    for root, dirs, files in os.walk(kaggle_input):
        if 'NORMAL' in dirs and 'PNEUMONIA' in dirs:
            DATA_DIR = os.path.dirname(root)
            break

print(f'Dataset: {DATA_DIR}')
print(f'Contents: {os.listdir(DATA_DIR)}')

TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VAL_DIR = os.path.join(DATA_DIR, 'val')
TEST_DIR = os.path.join(DATA_DIR, 'test')

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 2
CLASS_NAMES = ['NORMAL', 'PNEUMONIA']

In [ ]:
# CLAHE Preprocessing Function
def apply_clahe(img):
    """Apply CLAHE contrast enhancement to improve X-ray visibility"""
    if len(img.shape) == 3 and img.shape[2] == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    else:
        gray = img
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    enhanced_rgb = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2RGB)
    return enhanced_rgb


class CLAHEGenerator(tf.keras.utils.Sequence):
    def __init__(self, generator, apply_clahe_flag=True):
        self.generator = generator
        self.apply_clahe_flag = apply_clahe_flag
    def __len__(self):
        return len(self.generator)
    def __getitem__(self, idx):
        imgs, labels = self.generator[idx]
        if self.apply_clahe_flag:
            enhanced = []
            for img in imgs:
                img_uint8 = (img * 255).astype(np.uint8)
                enhanced_img = apply_clahe(img_uint8)
                enhanced.append(enhanced_img / 255.0)
            imgs = np.array(enhanced, dtype=np.float32)
        return imgs, labels
    def on_epoch_end(self):
        self.generator.on_epoch_end()

In [ ]:
# Data generators with stronger augmentation (NO CLAHE - it hurts this dataset)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2],
    shear_range=0.1,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='binary',
    classes=CLASS_NAMES, shuffle=True
)

val_gen = val_test_datagen.flow_from_directory(
    VAL_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='binary',
    classes=CLASS_NAMES, shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='binary',
    classes=CLASS_NAMES, shuffle=False
)

print(f'Train: {train_gen.samples} | Val: {val_gen.samples} | Test: {test_gen.samples}')

In [ ]:
# Visualize CLAHE effect
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
sample_batch = next(iter(train_gen))
sample_imgs, sample_labels = sample_batch

for i in range(5):
    axes[0, i].imshow(sample_imgs[i])
    axes[0, i].set_title(f'Original: {CLASS_NAMES[int(sample_labels[i])]}')
    axes[0, i].axis('off')
    img_uint8 = (sample_imgs[i] * 255).astype(np.uint8)
    enhanced = apply_clahe(img_uint8) / 255.0
    axes[1, i].imshow(enhanced)
    axes[1, i].set_title(f'CLAHE: {CLASS_NAMES[int(sample_labels[i])]}')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=12)
axes[1, 0].set_ylabel('CLAHE', fontsize=12)
plt.suptitle('CLAHE Preprocessing Effect', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# CBAM Attention Module
class ChannelAttention(layers.Layer):
    def __init__(self, reduction=16, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction
    def build(self, input_shape):
        channels = input_shape[-1]
        self.dense1 = layers.Dense(channels // self.reduction, activation='relu')
        self.dense2 = layers.Dense(channels, activation='sigmoid')
        self.global_avg = layers.GlobalAveragePooling2D()
        self.global_max = layers.GlobalMaxPooling2D()
    def call(self, x):
        avg_pool = self.global_avg(x)
        max_pool = self.global_max(x)
        avg_out = self.dense2(self.dense1(avg_pool))
        max_out = self.dense2(self.dense1(max_pool))
        attention = layers.Add()([avg_out, max_out])
        attention = layers.Activation('sigmoid')(attention)
        attention = layers.Reshape((1, 1, -1))(attention)
        return layers.Multiply()([x, attention])

class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size
    def build(self, input_shape):
        self.conv = layers.Conv2D(1, self.kernel_size, padding='same', activation='sigmoid')
    def call(self, x):
        avg_pool = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(x, axis=-1, keepdims=True)
        concat = layers.Concatenate()([avg_pool, max_pool])
        attention = self.conv(concat)
        return layers.Multiply()([x, attention])

class CBAM(layers.Layer):
    def __init__(self, reduction=16, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.channel_att = ChannelAttention(reduction)
        self.spatial_att = SpatialAttention(kernel_size)
    def call(self, x):
        x = self.channel_att(x)
        x = self.spatial_att(x)
        return x

In [ ]:
# Focal Loss
def focal_loss(gamma=2.0, alpha=0.25):
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, K.epsilon(), 1 - K.epsilon())
        bce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        focal_weight = alpha * tf.pow(1 - p_t, gamma)
        return tf.reduce_mean(focal_weight * bce)
    return loss_fn

In [ ]:
# Build improved MobileNetV2 + CBAM
def build_model():
    base_model = MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = CBAM(reduction=16, kernel_size=7)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs)
    return model, base_model

model, base_model = build_model()

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=focal_loss(gamma=2.0, alpha=0.25),
    metrics=['accuracy', keras.metrics.AUC(name='auc'),
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall')]
)

print(f'Total parameters: {model.count_params():,}')
model.summary()

In [ ]:
# Callbacks
cb_list = [
    callbacks.EarlyStopping(monitor='val_auc', patience=7, restore_best_weights=True, mode='max', verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1),
    callbacks.ModelCheckpoint('best_improved_model.keras', monitor='val_auc', save_best_only=True, mode='max', verbose=1)
]

In [ ]:
# Phase 1: Train classifier head (base frozen)
print('='*60)
print('Phase 1: Training classifier head + CBAM (base frozen)')
print('='*60)

history_p1 = model.fit(
    train_gen,
    epochs=12,
    validation_data=val_gen,
    callbacks=cb_list
)

print(f'\nPhase 1 Best Val AUC: {max(history_p1.history["val_auc"]):.4f}')
print(f'Phase 1 Best Val Acc: {max(history_p1.history["val_accuracy"]):.4f}')

In [ ]:
# Phase 2: Unfreeze top layers and fine-tune
print('='*60)
print('Phase 2: Fine-tuning with CBAM + Focal Loss')
print('='*60)

base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

print(f'Trainable layers: {sum(1 for l in base_model.layers if l.trainable)}')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=focal_loss(gamma=2.0, alpha=0.25),
    metrics=['accuracy', keras.metrics.AUC(name='auc'),
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall')]
)

history_p2 = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    callbacks=cb_list
)

print(f'\nPhase 2 Best Val AUC: {max(history_p2.history["val_auc"]):.4f}')
print(f'Phase 2 Best Val Acc: {max(history_p2.history["val_accuracy"]):.4f}')

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

all_train_loss = history_p1.history['loss'] + history_p2.history['loss']
all_val_loss = history_p1.history['val_loss'] + history_p2.history['val_loss']
all_train_acc = history_p1.history['accuracy'] + history_p2.history['accuracy']
all_val_acc = history_p1.history['val_accuracy'] + history_p2.history['val_accuracy']
all_train_auc = history_p1.history['auc'] + history_p2.history['auc']
all_val_auc = history_p1.history['val_auc'] + history_p2.history['val_auc']

axes[0].plot(all_train_loss, label='Train'); axes[0].plot(all_val_loss, label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(all_train_acc, label='Train'); axes[1].plot(all_val_acc, label='Val')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True)

axes[2].plot(all_train_auc, label='Train'); axes[2].plot(all_val_auc, label='Val')
axes[2].set_title('AUC'); axes[2].legend(); axes[2].grid(True)

plt.suptitle('Improved MobileNetV2 + CBAM + CLAHE + Focal Loss', fontsize=14)
plt.tight_layout()
plt.savefig('improved_training_history.png', dpi=150)
plt.show()

In [ ]:
# Evaluate on test set with CLAHE
predictions = model.predict(test_gen)
y_pred = (predictions > 0.5).astype(int).flatten()
y_true = test_gen.classes
y_probs = predictions.flatten()

print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Improved Model - Confusion Matrix')
plt.tight_layout()
plt.savefig('improved_confusion_matrix.png', dpi=150)
plt.show()

roc_auc = roc_auc_score(y_true, y_probs)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
accuracy = (y_pred == y_true).mean()

print(f'\nImproved Model Results:')
print(f'Accuracy:  {accuracy:.4f}')
print(f'ROC-AUC:   {roc_auc:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1-Score:  {f1:.4f}')

In [ ]:
# Compare with baseline
baseline = {'model': 'MobileNetV2 (Baseline)', 'accuracy': 0.8798, 'roc_auc': 0.9582,
             'precision': 0.8723, 'recall': 0.9462, 'f1_score': 0.9077}
improved = {'model': 'MobileNetV2+CBAM+CLAHE+Focal', 'accuracy': accuracy, 'roc_auc': roc_auc,
            'precision': precision, 'recall': recall, 'f1_score': f1}

print('\n' + '='*70)
print('MODEL COMPARISON: Baseline vs Improved')
print('='*70)
print(f'{"Metric":<15} {"Baseline":<15} {"Improved":<15} {"Change":<15}')
print('-'*70)
for metric in ['accuracy', 'roc_auc', 'precision', 'recall', 'f1_score']:
    base_val = baseline[metric]
    imp_val = improved[metric]
    change = imp_val - base_val
    sign = '+' if change > 0 else ''
    print(f'{metric:<15} {base_val:<15.4f} {imp_val:<15.4f} {sign}{change:<14.4f}')
print('='*70)

comparison = {'baseline': baseline, 'improved': improved,
              'improvements': ['CLAHE preprocessing', 'Focal Loss (gamma=2.0, alpha=0.25)',
                               'CBAM Attention (Channel + Spatial)', 'Stronger augmentation']}
with open('model_comparison.json', 'w') as f:
    json.dump(comparison, f, indent=2)
print('\nComparison saved to model_comparison.json')

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
metrics_list = ['accuracy', 'roc_auc', 'precision', 'recall', 'f1_score']
base_vals = [baseline[m] for m in metrics_list]
imp_vals = [improved[m] for m in metrics_list]
x = np.arange(len(metrics_list))
width = 0.35

axes[0].bar(x - width/2, base_vals, width, label='Baseline', color='#3498db')
axes[0].bar(x + width/2, imp_vals, width, label='Improved', color='#2ecc71')
axes[0].set_ylabel('Score'); axes[0].set_title('Baseline vs Improved')
axes[0].set_xticks(x)
axes[0].set_xticklabels(['Accuracy', 'AUC', 'Precision', 'Recall', 'F1'], rotation=45)
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3); axes[0].set_ylim(0.8, 1.0)

improvements_pct = [(imp_vals[i] - base_vals[i]) * 100 for i in range(len(metrics_list))]
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in improvements_pct]
axes[1].barh(metrics_list, improvements_pct, color=colors)
axes[1].set_xlabel('Improvement (%)'); axes[1].set_title('Improvement Over Baseline')
axes[1].axvline(x=0, color='black', linewidth=0.5); axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()

In [ ]:
# Save model and results
model.save('mobilenet_improved.keras')
model.save('mobilenet_improved.h5')

results = {
    'model': 'MobileNetV2 + CBAM + CLAHE + Focal Loss',
    'accuracy': float(accuracy), 'roc_auc': float(roc_auc),
    'precision': float(precision), 'recall': float(recall), 'f1_score': float(f1),
    'techniques': ['Focal Loss (gamma=2.0, alpha=0.25)',
                   'CBAM Attention', 'Stronger augmentation', 'Two-phase training'],
    'baseline_accuracy': 0.8798,
    'improvement': float(accuracy - 0.8798)
}
with open('improved_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'Model saved as mobilenet_improved.keras and .h5')
print(f'\nFinal Results: Accuracy={accuracy:.4f}, AUC={roc_auc:.4f}, F1={f1:.4f}')
print(f'Improvement over baseline: +{(accuracy - 0.8798)*100:.2f}% accuracy')